# Lab | Model Conversions & Inferencing

In [ ]:
import torch
import torch.nn as nn
from torchvision import models, transforms, datasets
from torch.utils.data import DataLoader
from PIL import Image
import numpy as np
import onnx
import onnxruntime as ort
import time
import os

device = 'cpu'
torch.manual_seed(42)

In [ ]:
model = models.resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, 102)
model.load_state_dict(torch.load('flowers102_resnet18.pth', map_location=device))
model.eval()
print('Model loaded.')

## Task 1 — Export to ONNX and Verify

### Part A — Export

In [ ]:
example = torch.randn(1, 3, 224, 224)

torch.onnx.export(
    model, example, 'flowers_resnet18.onnx',
    input_names=['input'], output_names=['logits'],
    dynamic_axes={'input': {0: 'batch'}, 'logits': {0: 'batch'}},
    opset_version=17,
)

onnx_model = onnx.load('flowers_resnet18.onnx')
onnx.checker.check_model(onnx_model)
print('ONNX model is valid.')

size_mb = os.path.getsize('flowers_resnet18.onnx') / 1e6
print(f'ONNX FP32 file size: {size_mb:.2f} MB')

### Part B — Numerical Equivalence Check

In [ ]:
val_tf = transforms.Compose([
    transforms.Resize(232),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

val_ds     = datasets.Flowers102(root='data', split='val', transform=val_tf, download=True)
val_loader = DataLoader(val_ds, batch_size=8, shuffle=False)

session = ort.InferenceSession('flowers_resnet18.onnx')

xb, _ = next(iter(val_loader))

with torch.no_grad():
    pt_out = model(xb).numpy()

ort_out = session.run(None, {'input': xb.numpy()})[0]

max_diff = np.abs(pt_out - ort_out).max()
print(f'Max absolute difference: {max_diff:.2e}')
assert max_diff < 1e-4, f'Equivalence check failed: {max_diff}'
print('Numerical equivalence confirmed.')

## Task 2 — Build an Inference Pipeline

In [ ]:
inference_code = '''
import numpy as np
import onnxruntime as ort
from PIL import Image


class FlowerClassifier:
    def __init__(self, onnx_path):
        self.session = ort.InferenceSession(onnx_path)
        self.mean = np.array([0.485, 0.456, 0.406], dtype=np.float32).reshape(1, 3, 1, 1)
        self.std  = np.array([0.229, 0.224, 0.225], dtype=np.float32).reshape(1, 3, 1, 1)

    def preprocess(self, image_path):
        img = Image.open(image_path).convert('RGB').resize((232, 232))
        left = (232 - 224) // 2
        img = img.crop((left, left, left + 224, left + 224))
        x = np.asarray(img, dtype=np.float32).transpose(2, 0, 1)[None] / 255.0
        return ((x - self.mean) / self.std).astype(np.float32)

    def predict(self, image_path, k=3):
        x = self.preprocess(image_path)
        logits = self.session.run(None, {'input': x})[0][0]
        probs = np.exp(logits - logits.max())
        probs /= probs.sum()
        top = np.argsort(probs)[::-1][:k]
        return [(int(i), float(probs[i])) for i in top]
'''

with open('inference.py', 'w') as f:
    f.write(inference_code.strip())
print('inference.py written.')

In [ ]:
from inference import FlowerClassifier

classifier = FlowerClassifier('flowers_resnet18.onnx')

test_ds_raw = datasets.Flowers102(root='data', split='test', download=True)
sample_paths = [test_ds_raw._image_files[i] for i in range(5)]

session_pt_check = ort.InferenceSession('flowers_resnet18.onnx')

for i, path in enumerate(sample_paths):
    preds = classifier.predict(str(path), k=3)
    print(f'Image {i+1}: top-3 = {preds}')

In [ ]:
from torchvision.transforms.functional import to_tensor, normalize, resize, center_crop

for path in sample_paths:
    img = Image.open(path).convert('RGB')
    img = img.resize((232, 232))
    left = (232 - 224) // 2
    img = img.crop((left, left, left + 224, left + 224))
    t = to_tensor(img).unsqueeze(0)
    t = normalize(t, [0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    with torch.no_grad():
        pt_logits = model(t).numpy()
    ort_logits = session_pt_check.run(None, {'input': t.numpy()})[0]
    diff = np.abs(pt_logits - ort_logits).max()
    print(f'{path.name} | PT top1={pt_logits[0].argmax()} | ORT top1={ort_logits[0].argmax()} | max_diff={diff:.2e}')

## Task 3 — Quantise to INT8 and Benchmark

In [ ]:
from onnxruntime.quantization import quantize_dynamic, QuantType

quantize_dynamic(
    model_input='flowers_resnet18.onnx',
    model_output='flowers_resnet18.int8.onnx',
    weight_type=QuantType.QInt8,
)

fp32_size = os.path.getsize('flowers_resnet18.onnx') / 1e6
int8_size = os.path.getsize('flowers_resnet18.int8.onnx') / 1e6
print(f'FP32 size : {fp32_size:.2f} MB')
print(f'INT8 size : {int8_size:.2f} MB')
print(f'Size ratio: {int8_size/fp32_size:.3f}x  ({fp32_size/int8_size:.1f}x smaller)')

In [ ]:
session_fp32 = ort.InferenceSession('flowers_resnet18.onnx')
session_int8 = ort.InferenceSession('flowers_resnet18.int8.onnx')

test_ds  = datasets.Flowers102(root='data', split='test', transform=val_tf, download=True)
test_loader_eval = DataLoader(test_ds, batch_size=32, shuffle=False)

def ort_accuracy(sess, loader):
    correct, total = 0, 0
    all_fp32, all_int8 = [], []
    for xb, yb in loader:
        xn = xb.numpy()
        out = sess.run(None, {'input': xn})[0]
        correct += (out.argmax(1) == yb.numpy()).sum()
        total += xb.size(0)
    return correct / total

acc_fp32 = ort_accuracy(session_fp32, test_loader_eval)
acc_int8 = ort_accuracy(session_int8, test_loader_eval)
print(f'FP32 ONNX test acc : {acc_fp32:.4f}')
print(f'INT8 ONNX test acc : {acc_int8:.4f}')
print(f'Accuracy drop      : {(acc_fp32 - acc_int8)*100:.2f}%')

In [ ]:
xb_bench = torch.randn(1, 3, 224, 224)
xb_np    = xb_bench.numpy()
N = 100

with torch.no_grad():
    _ = model(xb_bench)
t0 = time.perf_counter()
for _ in range(N):
    with torch.no_grad():
        _ = model(xb_bench)
lat_pt = (time.perf_counter() - t0) / N * 1000

_ = session_fp32.run(None, {'input': xb_np})
t0 = time.perf_counter()
for _ in range(N):
    _ = session_fp32.run(None, {'input': xb_np})
lat_fp32 = (time.perf_counter() - t0) / N * 1000

_ = session_int8.run(None, {'input': xb_np})
t0 = time.perf_counter()
for _ in range(N):
    _ = session_int8.run(None, {'input': xb_np})
lat_int8 = (time.perf_counter() - t0) / N * 1000

pt_size_mb = sum(p.numel() * 4 for p in model.parameters()) / 1e6

print(f'{'Model':<20} {'Size (MB)':>12} {'Avg lat (ms)':>14} {'Speedup':>10}')
print('-'*60)
print(f'{'PyTorch (FP32)':<20} {pt_size_mb:>12.2f} {lat_pt:>14.2f} {'1.00x':>10}')
print(f'{'ONNX (FP32)':<20} {fp32_size:>12.2f} {lat_fp32:>14.2f} {lat_pt/lat_fp32:>9.2f}x')
print(f'{'ONNX (INT8)':<20} {int8_size:>12.2f} {lat_int8:>14.2f} {lat_pt/lat_int8:>9.2f}x')

## Benchmark Commentary

ONNX Runtime FP32 is typically 1.5–2× faster than PyTorch on CPU for single-image inference because
it eliminates Python overhead and applies graph-level optimisations (operator fusion, memory layout
reordering) that PyTorch eager mode cannot.
INT8 dynamic quantisation compresses weight matrices from 32-bit floats to 8-bit integers, cutting
model size by roughly 4× and reducing memory bandwidth, which is the dominant bottleneck on modern
CPUs — hence an additional 1.5–2× latency improvement on top of the FP32 ONNX baseline.
Most of the gain therefore comes from removing Python overhead (PyTorch → ONNX FP32) and then from
reduced memory reads (FP32 ONNX → INT8 ONNX).
